# Synthetic Satellite Downlink PSD Dataset Generator (SciPy)

This notebook generates labeled Power Spectral Density (PSD) plots of synthetic satellite downlink signals using NumPy/SciPy.

## Overview
- Pure Python implementation (no GNURadio dependency)
- Generate multi-carrier satellite downlink scenarios  
- Support DVB-S2 modulation schemes (QPSK, 8PSK, 16APSK, 32APSK)
- Compute and visualize PSDs with labeled carrier parameters
- Export dataset for ML training

## 1. Imports & Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import json
import os
from datetime import datetime
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ All imports successful")

✓ All imports successful


## 2. Parameter Configuration

In [2]:
# Dataset generation parameters
NUM_SAMPLES = 100  # Total number of PSD samples to generate
MIN_CARRIERS = 1    # Minimum carriers per sample
MAX_CARRIERS = 5    # Maximum carriers per sample

# RF parameters (Ku-band satellite downlink)
SAMPLE_RATE = 100e6  # 100 MHz sampling rate
FREQ_SPAN = 80e6     # 80 MHz frequency span

# Carrier parameter ranges
PARAM_RANGES = {
    'bandwidth_mhz': (1, 36),           # Transponder bandwidth
    'modulation': ['QPSK', '8PSK', '16APSK', '32APSK'],
    'roll_off': [0.20, 0.25, 0.35],     # DVB-S2 standard roll-offs
    'code_rate': ['1/4', '1/3', '2/5', '1/2', '3/5', '2/3', '3/4', '4/5', '5/6', '8/9', '9/10'],
    'power_dbm': (-80, -40),            # Carrier power
    'cn_db': (-2.4, 16.0),              # Carrier-to-noise ratio
}

# Output paths
DATA_DIR = Path('../../data/raw')
PSD_IMG_DIR = DATA_DIR / 'psd_images_scipy'
PSD_ARRAY_DIR = DATA_DIR / 'psd_arrays_scipy'
LABELS_FILE = DATA_DIR / 'labels_scipy.json'

# Create directories
for d in [PSD_IMG_DIR, PSD_ARRAY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dataset configuration:")
print(f"  Samples: {NUM_SAMPLES}")
print(f"  Carriers per sample: {MIN_CARRIERS}-{MAX_CARRIERS}")
print(f"  Sample rate: {SAMPLE_RATE/1e6:.0f} MHz")
print(f"  Frequency span: {FREQ_SPAN/1e6:.0f} MHz")

Dataset configuration:
  Samples: 100
  Carriers per sample: 1-5
  Sample rate: 100 MHz
  Frequency span: 80 MHz


## 3. Modulation & Pulse Shaping Functions

In [3]:
def get_constellation(mod_type):
    """
    Return constellation points for different modulation schemes.
    """
    if mod_type == 'QPSK':
        # QPSK: 4 points at 45, 135, 225, 315 degrees
        return np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2)
    
    elif mod_type == '8PSK':
        # 8PSK: 8 equally spaced points on unit circle
        angles = np.arange(8) * 2 * np.pi / 8 + np.pi/8
        return np.exp(1j * angles)
    
    elif mod_type == '16APSK':
        # 16APSK: DVB-S2 constellation (4+12 ring configuration)
        # Inner ring: 4 points
        r1 = 1.0
        inner = r1 * np.exp(1j * np.arange(4) * 2 * np.pi / 4 + np.pi/4)
        
        # Outer ring: 12 points
        r2 = 2.7  # Typical ratio
        outer = r2 * np.exp(1j * np.arange(12) * 2 * np.pi / 12 + np.pi/12)
        
        constellation = np.concatenate([inner, outer])
        # Normalize to unit average power
        return constellation / np.sqrt(np.mean(np.abs(constellation)**2))
    
    elif mod_type == '32APSK':
        # 32APSK: DVB-S2 constellation (4+12+16 ring configuration)
        # Inner ring: 4 points
        r1 = 1.0
        inner = r1 * np.exp(1j * np.arange(4) * 2 * np.pi / 4 + np.pi/4)
        
        # Middle ring: 12 points
        r2 = 2.7
        middle = r2 * np.exp(1j * np.arange(12) * 2 * np.pi / 12 + np.pi/12)
        
        # Outer ring: 16 points
        r3 = 4.3
        outer = r3 * np.exp(1j * np.arange(16) * 2 * np.pi / 16)
        
        constellation = np.concatenate([inner, middle, outer])
        return constellation / np.sqrt(np.mean(np.abs(constellation)**2))
    
    else:
        # Default to QPSK
        return np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2)


def rrc_filter(symbol_rate, sample_rate, roll_off, span=11):
    """
    Generate root-raised-cosine (RRC) filter taps for pulse shaping.
    
    Args:
        symbol_rate: Symbol rate (sps)
        sample_rate: Sample rate (Hz)
        roll_off: Roll-off factor (beta)
        span: Filter span in symbols
    
    Returns:
        np.array: Filter taps
    """
    sps = int(sample_rate / symbol_rate)  # Samples per symbol
    num_taps = span * sps + 1
    
    # Time vector
    t = np.arange(-span*sps//2, span*sps//2 + 1) / sample_rate
    
    # RRC formula
    taps = np.zeros(num_taps)
    for i, time in enumerate(t):
        if time == 0:
            taps[i] = (1 + roll_off * (4/np.pi - 1))
        elif abs(time) == 1 / (4 * roll_off * symbol_rate):
            taps[i] = (roll_off / np.sqrt(2)) * \
                     ((1 + 2/np.pi) * np.sin(np.pi/(4*roll_off)) + \
                      (1 - 2/np.pi) * np.cos(np.pi/(4*roll_off)))
        else:
            num = np.sin(np.pi * time * symbol_rate * (1 - roll_off)) + \
                  4 * roll_off * time * symbol_rate * np.cos(np.pi * time * symbol_rate * (1 + roll_off))
            den = np.pi * time * symbol_rate * (1 - (4 * roll_off * time * symbol_rate)**2)
            taps[i] = num / den
    
    # Normalize
    taps /= np.sqrt(np.sum(taps**2))
    
    return taps

print("✓ Modulation and pulse shaping functions defined")

✓ Modulation and pulse shaping functions defined


## 4. Signal Generation Functions

In [4]:
def generate_carrier(center_freq, bandwidth, modulation, symbol_rate, 
                    roll_off, power_dbm, sample_rate, num_samples):
    """
    Generate a single modulated carrier with pulse shaping.
    
    Returns:
        np.array: Complex baseband signal
    """
    # Get constellation
    constellation = get_constellation(modulation)
    
    # Calculate samples per symbol
    sps = int(sample_rate / symbol_rate)
    num_symbols = (num_samples // sps) + 20  # Extra symbols for filtering
    
    # Generate random symbols
    symbol_indices = np.random.randint(0, len(constellation), num_symbols)
    symbols = constellation[symbol_indices]
    
    # Upsample (zero-stuff)
    upsampled = np.zeros(num_symbols * sps, dtype=complex)
    upsampled[::sps] = symbols
    
    # Root-raised-cosine pulse shaping
    rrc_taps = rrc_filter(symbol_rate, sample_rate, roll_off, span=11)
    shaped = signal.lfilter(rrc_taps, 1, upsampled)
    
    # Trim to desired length
    shaped = shaped[:num_samples]
    
    # Frequency shift to center frequency
    t = np.arange(len(shaped)) / sample_rate
    carrier_sig = shaped * np.exp(2j * np.pi * center_freq * t)
    
    # Apply power scaling (convert dBm to linear)
    power_linear = 10 ** (power_dbm / 10)  # mW
    carrier_sig *= np.sqrt(power_linear / np.mean(np.abs(carrier_sig)**2))
    
    return carrier_sig


def add_noise(signal_data, cn_db):
    """
    Add AWGN to achieve target C/N ratio.
    
    Args:
        signal_data: Complex signal
        cn_db: Carrier-to-Noise ratio in dB
    
    Returns:
        np.array: Signal with noise added
    """
    signal_power = np.mean(np.abs(signal_data) ** 2)
    cn_linear = 10 ** (cn_db / 10)
    noise_power = signal_power / cn_linear
    
    # Generate complex AWGN
    noise_std = np.sqrt(noise_power / 2)
    noise = noise_std * (np.random.randn(len(signal_data)) + 
                         1j * np.random.randn(len(signal_data)))
    
    return signal_data + noise


def combine_carriers(carrier_params, sample_rate, num_samples):
    """
    Generate and combine multiple carriers.
    
    Args:
        carrier_params: List of dictionaries with carrier parameters
        
    Returns:
        np.array: Combined complex signal
    """
    combined = np.zeros(num_samples, dtype=complex)
    
    for params in carrier_params:
        carrier = generate_carrier(
            center_freq=params['center_freq_hz'],
            bandwidth=params['bandwidth_hz'],
            modulation=params['modulation'],
            symbol_rate=params['symbol_rate_sps'],
            roll_off=params['roll_off'],
            power_dbm=params['power_dbm'],
            sample_rate=sample_rate,
            num_samples=num_samples
        )
        combined += carrier
    
    # Add noise based on average C/N
    avg_cn = np.mean([p['cn_ratio_db'] for p in carrier_params])
    combined = add_noise(combined, avg_cn)
    
    return combined

print("✓ Signal generation functions defined")

✓ Signal generation functions defined


## 5. PSD Computation

In [5]:
def compute_psd(signal_data, sample_rate, nperseg=1024):
    """
    Compute Power Spectral Density using Welch's method.
    
    Returns:
        freqs: Frequency bins (Hz)
        psd_db: PSD in dB scale
    """
    freqs, psd = signal.welch(signal_data, fs=sample_rate, nperseg=nperseg,
                              return_onesided=False, scaling='density')
    
    # Shift to center DC at middle
    freqs = np.fft.fftshift(freqs)
    psd = np.fft.fftshift(psd)
    
    # Convert to dB
    psd_db = 10 * np.log10(psd + 1e-12)  # Add small value to avoid log(0)
    
    return freqs, psd_db

print("✓ PSD computation function defined")

✓ PSD computation function defined


## 6. Visualization

In [6]:
def plot_psd(freqs, psd_db, carrier_params, save_path=None, show=True):
    """
    Plot PSD with carrier annotations.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot PSD
    ax.plot(freqs / 1e6, psd_db, linewidth=0.8, color='blue', alpha=0.7)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power Spectral Density (dB/Hz)')
    ax.set_title('Satellite Downlink PSD with Labeled Carriers')
    ax.grid(True, alpha=0.3)
    
    # Annotate carriers
    colors = plt.cm.Set3(np.linspace(0, 1, len(carrier_params)))
    
    for idx, params in enumerate(carrier_params):
        fc = params['center_freq_hz'] / 1e6  # MHz
        bw = params['bandwidth_hz'] / 1e6    # MHz
        
        # Draw carrier span
        ax.axvspan(fc - bw/2, fc + bw/2, alpha=0.2, color=colors[idx])
        
        # Add label
        label = f"C{idx}: {params['modulation']}\n{params['symbol_rate_sps']/1e6:.1f} Msps\nα={params['roll_off']}"
        y_offset = ax.get_ylim()[1] - 5 - idx*4
        ax.text(fc, y_offset, label, 
                ha='center', fontsize=7, bbox=dict(boxstyle='round', 
                facecolor=colors[idx], alpha=0.6))
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig

print("✓ Visualization function defined")

✓ Visualization function defined


## 7. Carrier Parameter Sampling

In [7]:
def sample_carrier_params(freq_span, sample_rate, carrier_id):
    """
    Sample random carrier parameters from defined ranges.
    """
    # Bandwidth
    bw_mhz = np.random.uniform(*PARAM_RANGES['bandwidth_mhz'])
    bw_hz = bw_mhz * 1e6
    
    # Center frequency (keep away from edges)
    margin = bw_hz
    fc_hz = np.random.uniform(-freq_span/2 + margin, freq_span/2 - margin)
    
    # Modulation and other params
    modulation = np.random.choice(PARAM_RANGES['modulation'])
    roll_off = np.random.choice(PARAM_RANGES['roll_off'])
    code_rate = np.random.choice(PARAM_RANGES['code_rate'])
    power_dbm = np.random.uniform(*PARAM_RANGES['power_dbm'])
    cn_db = np.random.uniform(*PARAM_RANGES['cn_db'])
    
    # Symbol rate from bandwidth and roll-off
    symbol_rate = bw_hz / (1 + roll_off)
    
    return {
        'id': carrier_id,
        'center_freq_hz': float(fc_hz),
        'bandwidth_hz': float(bw_hz),
        'modulation': modulation,
        'symbol_rate_sps': float(symbol_rate),
        'roll_off': float(roll_off),
        'code_rate': code_rate,
        'power_dbm': float(power_dbm),
        'cn_ratio_db': float(cn_db)
    }

print("✓ Parameter sampling function defined")

✓ Parameter sampling function defined


## 8. Dataset Generation Loop

In [8]:
def generate_dataset(num_samples, min_carriers, max_carriers):
    """
    Main dataset generation function.
    """
    dataset = []
    num_time_samples = int(SAMPLE_RATE * 0.01)  # 10ms of data
    
    for sample_idx in range(num_samples):
        # Random number of carriers
        num_carriers = np.random.randint(min_carriers, max_carriers + 1)
        
        # Sample carrier parameters
        carriers = [sample_carrier_params(FREQ_SPAN, SAMPLE_RATE, i) 
                    for i in range(num_carriers)]
        
        # Generate combined signal
        signal_data = combine_carriers(carriers, SAMPLE_RATE, num_time_samples)
        
        # Compute PSD
        freqs, psd_db = compute_psd(signal_data, SAMPLE_RATE)
        
        # Save PSD plot
        img_path = PSD_IMG_DIR / f'psd_{sample_idx:05d}.png'
        plot_psd(freqs, psd_db, carriers, save_path=img_path, show=False)
        
        # Save PSD array
        array_path = PSD_ARRAY_DIR / f'psd_{sample_idx:05d}.npy'
        np.save(array_path, {'freqs': freqs, 'psd_db': psd_db})
        
        # Store metadata
        metadata = {
            'sample_id': sample_idx,
            'psd_image': str(img_path.relative_to(DATA_DIR)),
            'psd_array': str(array_path.relative_to(DATA_DIR)),
            'sample_rate': float(SAMPLE_RATE),
            'frequency_span': float(FREQ_SPAN),
            'num_carriers': num_carriers,
            'carriers': carriers,
            'generation_timestamp': datetime.now().isoformat(),
            'method': 'scipy'
        }
        dataset.append(metadata)
        
        if (sample_idx + 1) % 10 == 0:
            print(f"Generated {sample_idx + 1}/{num_samples} samples")
    
    return dataset

print("✓ Dataset generation function defined")

✓ Dataset generation function defined


## 9. Test Single Carrier Generation

In [9]:
# Test with a single carrier first
print("Testing single carrier generation...\n")

test_params = sample_carrier_params(FREQ_SPAN, SAMPLE_RATE, 0)
print("Test carrier parameters:")
for key, value in test_params.items():
    if 'hz' in key.lower() or 'sps' in key.lower():
        print(f"  {key}: {value/1e6:.2f} MHz" if 'hz' in key.lower() else f"  {key}: {value/1e6:.2f} Msps")
    else:
        print(f"  {key}: {value}")

# Generate signal
num_samples = int(SAMPLE_RATE * 0.01)  # 10ms
test_signal = combine_carriers([test_params], SAMPLE_RATE, num_samples)

# Compute and plot PSD
freqs, psd_db = compute_psd(test_signal, SAMPLE_RATE)
plot_psd(freqs, psd_db, [test_params], show=True)

print("\n✓ Single carrier test successful")

Testing single carrier generation...

Test carrier parameters:
  id: 0
  center_freq_hz: 23.34 MHz
  bandwidth_hz: 14.11 MHz
  modulation: 16APSK
  symbol_rate_sps: 11.76 Msps
  roll_off: 0.2
  code_rate: 3/5
  power_dbm: -73.75925438230254
  cn_ratio_db: 0.47029917418612843


/tmp/ipykernel_561567/177041922.py:31: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()



✓ Single carrier test successful


## 10. Generate Full Dataset

In [ ]:
print(f"Starting dataset generation...")
print(f"This will generate {NUM_SAMPLES} samples with {MIN_CARRIERS}-{MAX_CARRIERS} carriers each.\n")

dataset = generate_dataset(NUM_SAMPLES, MIN_CARRIERS, MAX_CARRIERS)

print(f"\n✓ Generated {len(dataset)} samples")

## 11. Save Labels

In [ ]:
with open(LABELS_FILE, 'w') as f:
    json.dump(dataset, f, indent=2)

print(f"✓ Saved labels to {LABELS_FILE}")
print(f"  Total size: {LABELS_FILE.stat().st_size / 1024:.1f} KB")

## 12. Validation & Statistics

In [ ]:
# Analyze dataset
import pandas as pd

# Extract carrier statistics
all_carriers = []
for sample in dataset:
    all_carriers.extend(sample['carriers'])

df = pd.DataFrame(all_carriers)

print("Dataset Statistics:")
print(f"  Total samples: {len(dataset)}")
print(f"  Total carriers: {len(all_carriers)}")
print(f"  Avg carriers per sample: {len(all_carriers)/len(dataset):.2f}\n")

print("Modulation distribution:")
print(df['modulation'].value_counts())
print("\nRoll-off distribution:")
print(df['roll_off'].value_counts())
print("\nCode rate distribution:")
print(df['code_rate'].value_counts())

# Plot parameter distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df['bandwidth_hz']/1e6, bins=20, edgecolor='black')
axes[0, 0].set_xlabel('Bandwidth (MHz)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Bandwidth Distribution')

axes[0, 1].hist(df['power_dbm'], bins=20, edgecolor='black')
axes[0, 1].set_xlabel('Power (dBm)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Power Distribution')

axes[1, 0].hist(df['cn_ratio_db'], bins=20, edgecolor='black')
axes[1, 0].set_xlabel('C/N Ratio (dB)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('C/N Distribution')

df['modulation'].value_counts().plot(kind='bar', ax=axes[1, 1], edgecolor='black')
axes[1, 1].set_xlabel('Modulation Type')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Modulation Distribution')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 13. Visualize Sample PSDs

In [ ]:
# Display a few random examples
sample_indices = np.random.choice(len(dataset), min(3, len(dataset)), replace=False)

for idx in sample_indices:
    sample = dataset[idx]
    img_path = DATA_DIR / sample['psd_image']
    
    print(f"\nSample {idx}: {sample['num_carriers']} carriers")
    for carrier in sample['carriers']:
        print(f"  - {carrier['modulation']}: {carrier['symbol_rate_sps']/1e6:.1f} Msps, "
              f"BW={carrier['bandwidth_hz']/1e6:.1f} MHz, "
              f"FC={carrier['center_freq_hz']/1e6:.1f} MHz, "
              f"α={carrier['roll_off']:.2f}")
    
    # Display image
    from IPython.display import Image, display
    display(Image(filename=str(img_path)))

## Summary

This notebook uses pure NumPy/SciPy to generate synthetic satellite downlink PSDs with labeled carriers. The implementation includes:

- Full constellation mappings for QPSK, 8PSK, 16APSK, 32APSK
- Root-raised-cosine pulse shaping
- Realistic carrier power and C/N ratios
- Multi-carrier scenarios with frequency overlap

The dataset is saved to `data/raw/` with:
- PSD plots as PNG images
- PSD arrays as NumPy files
- Metadata labels in JSON format

Next steps:
1. Split data into train/val/test sets
2. Build ML model for carrier detection/classification
3. Train and evaluate model